In [ ]:
(* load the setup of the notebooks *)
#use "../setup.ml";;

## Utility Functions

The function `range` takes two arguments:
 * `start` is an optional argument with the default value 0,
 * `stop` is an integer.

It returns the list $[\texttt{start}, \cdots, \texttt{stop}]$.  Note that `stop` is included.  The TypeScript version *overloads* the function `range`, i.e. it defines two functions with the same name but a different number of arguments.  <samp>OCaml</samp> does not support overloading, but the optional argument has the same effect: `range 3` creates the list `[0; 1; 2; 3]`, while `range ~start:2 5` creates the list `[2; 3; 4; 5]`.

In [ ]:
let range ?(start = 0) (stop : int) : int list =
  List.init (max 0 (stop - start + 1)) (fun i -> start + i)

In [ ]:
range 3

In [ ]:
range ~start:2 5

# The Knight's Tour

This notebook computes a solution to the [knight's tour](https://en.wikipedia.org/wiki/Knight%27s_tour) using the constraint solver `Z3`.  

In [ ]:
import_file "../z3_helpers.ml"

The function `row` takes one argument:
- `i` is an integer from the set $\{0, 1, \cdots, 63\}$.

It computes the name of the variable that specifies the *row* of the knight after its $i^{\textrm{th}}$ move.

In [ ]:
let row (i : int) : string = Printf.sprintf "R%d" i

In [ ]:
row 1

The function `col` takes one argument:
- `i` is an integer from the set $\{0, 1, \cdots, 63\}$.

It computes the name of the variable that specifies the *column* of the knight after its $i^{\textrm{th}}$ move.

In [ ]:
let col (i : int) : string = Printf.sprintf "C%d" i

The function `to_bitvar` takes one argument:
- `x` is a string.

It turns `x` into a variable whose type is the type of bit vectors of length 4.

In [ ]:
let to_bitvar (x : string) : Z3.Expr.expr = bitvec_const x 4

The function `is_knight_move` takes four arguments:
* `row` is a `Z3` variable that specifies the row of the position of the knight before the move.
* `col` is a `Z3` variable that specifies the column of the position of the knight before the move.
* `row_x` is a `Z3` variable that specifies the row of the position of the knight after the move.
* `col_x` is a `Z3` variable that specifies the column of the position of the knight after the move.

It returns a formula that specifies that the specified position represents a legal move for a knight.

The formula is the disjunction of the eight possible moves.  The moves are written as strings, in which the names `R`, `C`, `R'`, and `C'` stand for `row`, `col`, `row_x`, and `col_x`.  `String.concat " || " moves` joins them to a disjunction.  As the variables are bit vectors of length 4, the numbers in the formulas are bit vectors of length 4, too, and the arithmetic is computed modulo $2^4 = 16$.  Therefore, `C - 2` is the same as `C + 14`.  If a move leaves the board, the new row or column is at least 8, and the move is excluded by the constraints of `all_constraints`.

The figure below shows the moves of a knight:  The knight on `e4` can jump to all red squares.
<img src="knight-moves.png" width="50%">

In [ ]:
let is_knight_move (row : Z3.Expr.expr) (col : Z3.Expr.expr)
                   (row_x : Z3.Expr.expr) (col_x : Z3.Expr.expr) : Z3.Expr.expr =
  let moves = ["R' = R + 1 && C' = C + 2"; "R' = R + 1 && C' = C - 2";
               "R' = R + 2 && C' = C + 1"; "R' = R + 2 && C' = C - 1";
               "R' = R - 1 && C' = C + 2"; "R' = R - 1 && C' = C - 2";
               "R' = R - 2 && C' = C + 1"; "R' = R - 2 && C' = C - 1"] in
  expr ~vars:["R", row; "C", col; "R'", row_x; "C'", col_x] (String.concat " || " moves)

The function `all_different` takes two arguments:
* `rows` is an array of `Z3` variables.  The variable `rows.(i)` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `cols` is an array of `Z3` variables.  The variable `cols.(i)` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

The function returns a list of formulas stating that for $i \not= j$ the position after the $i^{\textrm{th}}$ move
differs from the position after the $j^{\textrm{th}}$ move.  In the formula, the names `Ri`, `Ci`, `Rj`, and `Cj` stand for `rows.(i)`, `cols.(i)`, `rows.(j)`, and `cols.(j)`.

In [ ]:
let all_different (rows : Z3.Expr.expr array) (cols : Z3.Expr.expr array) : Z3.Expr.expr list =
  List.concat_map
    (fun i ->
       List.map
         (fun j ->
            expr ~vars:["Ri", rows.(i); "Ci", cols.(i); "Rj", rows.(j); "Cj", cols.(j)]
              "Ri <> Rj || Ci <> Cj")
         (range ~start:(i + 1) 63))
    (range 62)

The function `all_constraints` takes two arguments:
* `rows` is an array of `Z3` variables.  The variable `rows.(i)` specifies the row of the position of the knight after the $i^{\textrm{th}}$ move.
* `cols` is an array of `Z3` variables.  The variable `cols.(i)` specifies the column of the position of the knight after the $i^{\textrm{th}}$ move.

It returns a list containing all constraints of the problem:  The positions are all different, the knight starts in the upper left corner, every move is a knight move, and all positions are on the board.  Bit vectors are compared as unsigned numbers.

In [ ]:
let all_constraints (rows : Z3.Expr.expr array) (cols : Z3.Expr.expr array) : Z3.Expr.expr list =
  all_different rows cols
  @ constraints ~vars:["R", rows.(0); "C", cols.(0)] ["R = 0"; "C = 0"]
  @ List.map (fun i -> is_knight_move rows.(i) cols.(i) rows.(i + 1) cols.(i + 1)) (range 62)
  @ List.concat_map (fun i -> constraints ~vars:["R", rows.(i); "C", cols.(i)] ["R < 8"; "C < 8"]) (range 63)

The function `solve` takes the argument `()`.  It computes a solution of the knight's problem.  The result is either `Some solution`, where `solution` is the list of the positions `(row, col)` of the knight after every move, or `None`.  The function `eval_bitvec` converts the value of a bit vector into an <samp>OCaml</samp> integer.

**Implementation:** Z3 makes random choices while it searches for a solution.  These choices are determined by the parameter `random_seed`, which is 0 by default.  For this problem, the seed 13 leads to a solution much faster than the default.  The parameter is set with a parameter set of type `Z3.Params.params`, which is created by `Z3.Params.mk_params` and passed to the solver via `Z3.Solver.set_parameters`.  Parameter names are Z3 *symbols*, which are created by `Z3.Symbol.mk_string`.

In [ ]:
let solve () : (int * int) list option =
  let rows = Array.of_list (List.map (fun i -> to_bitvar (row i)) (range 63)) in
  let cols = Array.of_list (List.map (fun i -> to_bitvar (col i)) (range 63)) in
  let solver = new_solver () in
  (* With the seed 13, Z3 finds a solution in about a minute.  Without this parameter,
     it needs about 15 minutes on a Mac with an M1 processor. *)
  let params = Z3.Params.mk_params ctx in
  Z3.Params.add_int params (Z3.Symbol.mk_string ctx "random_seed") 13;
  Z3.Solver.set_parameters solver params;
  Z3.Solver.add solver (all_constraints rows cols);
  match Z3.Solver.check solver [] with
  | Z3.Solver.SATISFIABLE ->
      let model = solver_model solver in
      Some (List.map (fun i -> (eval_bitvec model rows.(i), eval_bitvec model cols.(i))) (range 63))
  | Z3.Solver.UNSATISFIABLE ->
      print_endline "The problem is not solvable.";
      None
  | Z3.Solver.UNKNOWN ->
      print_endline "Z3 cannot determine whether the problem is solvable.";
      None

Unfortunately, the execution time of the following cell is hard to predict.  It depends on the random choices that Z3 makes while searching for a solution.  The TypeScript version of this notebook needs between one and a half and 30 minutes.  For a given seed, Z3 makes the same choices in every run, so the time hardly varies from run to run.  On my Mac Studio from 2022 (M1 Max), the <samp>OCaml</samp> version needs about a minute with the seed 13 and about 15 minutes without it.  In a test of the seeds 0 to 17, the seed 13 was the fastest; with the other seeds, Z3 needed between one and more than five minutes.  Which seed is fast also depends on the environment: When the same code is run by the toplevel `ocaml` instead of the Jupyter kernel, other seeds are fast.  Hence, with other versions of Z3 or of the <samp>OCaml</samp> kernel, a different seed may be needed.

In [ ]:
let solution = time "Solver Duration" solve

The function `create_board` takes one argument:
- `solution` is the list of positions computed by `solve`.

It returns a matrix `board` of size $8\times 8$.
The following holds:
$$ \texttt{board}[\texttt{R}i][\texttt{C}i] = i $$
Therefore, if `board.(r).(c) = i`, then at the beginning of the $i^{\textrm{th}}$ move the knight is located in row `r` and column `c`.  `Array.make_matrix 8 8 0` creates an $8 \times 8$ matrix filled with zeros, and `List.iteri` passes the index of every position to the function that processes it.

In [ ]:
let create_board (solution : (int * int) list) : int array array =
  let board = Array.make_matrix 8 8 0 in
  List.iteri (fun i (r, c) -> board.(r).(c) <- i) solution;
  board

In [ ]:
let board = create_board (Option.get solution)

The function `print_board` takes one argument:
- `board` is a square matrix of numbers.

It prints `board` as a table with frame lines.  All numbers are right aligned in columns of the same width.

In [ ]:
let print_board (board : int array array) : unit =
  let n = Array.length board in
  if n > 0 then begin
    let width =
      Array.fold_left (Array.fold_left (fun m x -> max m (String.length (string_of_int x)))) 0 board
    in
    let repeat s k = String.concat "" (List.init k (fun _ -> s)) in
    let create_line left mid right fill =
      left ^ String.concat mid (List.init n (fun _ -> repeat fill (width + 2))) ^ right
    in
    print_endline (create_line "╔" "╦" "╗" "═");
    Array.iteri
      (fun i r ->
         print_endline
           ("║" ^ String.concat "" (Array.to_list (Array.map (fun x -> Printf.sprintf " %*d ║" width x) r)));
         if i < n - 1 then print_endline (create_line "╠" "╬" "╣" "═"))
      board;
    print_endline (create_line "╚" "╩" "╝" "═")
  end

In [ ]:
print_board board

# Visualization

The function `show_solution` takes two arguments:
- `board` is the matrix computed by `create_board`.  We have `board.(row).(col) = k` if the $k^\textrm{th}$ move leads the knight to the position `(row, col)`.
- `width` is the width of the picture.  It is an optional argument with the default value `"50%"`.

It displays the given solution on a chessboard.  The button starts an animation that shows the path of the knight.

**Implementation:** The board, the numbers, and the path are created as SVG elements.  The lines and circles of the path are initially invisible.  The script that makes them visible one after the other is stored in the quoted string `script`, which contains the placeholders `__UID__` and `__MAXSTEPS__`.  `__UID__` is replaced by a random identifier, so that the elements of different animations in the same notebook have different names.  The function `replace_all s pattern by` replaces every occurrence of the string `pattern` in `s` by the string `by`.

In [ ]:
#require "jupyter.notebook";;

In [ ]:
let replace_all (s : string) (pattern : string) (by : string) : string =
  let n = String.length s and k = String.length pattern in
  let buffer = Buffer.create n in
  let rec loop i =
    if i > n - k then
      Buffer.add_string buffer (String.sub s i (n - i))
    else if String.sub s i k = pattern then begin
      Buffer.add_string buffer by;
      loop (i + k)
    end else begin
      Buffer.add_char buffer s.[i];
      loop (i + 1)
    end
  in
  loop 0;
  Buffer.contents buffer

let button = {|<button id="btn-__UID__" onclick="playKnightTour___UID__()" style="padding: 10px 15px; margin-bottom: 10px; cursor: pointer; background: #4CAF50; color: white; border: none; border-radius: 4px; font-size: 14px;">▶ Play Epic Tour</button><br>|}

let script = {|
    <script>
        let isPlaying___UID__ = false;

        function playKnightTour___UID__() {
            if (isPlaying___UID__) return;
            isPlaying___UID__ = true;
            
            const btn = document.getElementById('btn-__UID__');
            btn.style.background = '#888';
            btn.innerText = 'Running...';

            const maxSteps = __MAXSTEPS__;
            const delay = 500; // Half speed (was 250ms)
            
            const AudioContext = window.AudioContext || window.webkitAudioContext;
            const audioCtx = new AudioContext();
            if (audioCtx.state === 'suspended') audioCtx.resume();
            
            // --- 1. SETUP REVERB ---
            const convolver = audioCtx.createConvolver();
            const rate = audioCtx.sampleRate;
            const length = rate * 2.0; // 2 seconds of reverb decay
            const impulse = audioCtx.createBuffer(2, length, rate);
            for (let i = 0; i < length; i++) {
                const decay = Math.exp(-i / (rate * 0.3)); // Reverb tail
                impulse.getChannelData(0)[i] = (Math.random() * 2 - 1) * decay;
                impulse.getChannelData(1)[i] = (Math.random() * 2 - 1) * decay;
            }
            convolver.buffer = impulse;
            convolver.connect(audioCtx.destination);

            // --- 2. SOUND DESIGN FUNCTIONS ---
            function playNeigh() {
                const osc = audioCtx.createOscillator();
                const lfo = audioCtx.createOscillator();
                const tremolo = audioCtx.createGain();
                const mainGain = audioCtx.createGain();
                
                // The base sound: A descending sawtooth
                osc.type = 'sawtooth';
                osc.frequency.setValueAtTime(900, audioCtx.currentTime);
                osc.frequency.exponentialRampToValueAtTime(300, audioCtx.currentTime + 1.2);
                
                // Tremolo effect to simulate the "chatter" of a whinny
                lfo.type = 'sine';
                lfo.frequency.value = 18; 
                tremolo.gain.value = 0.8;
                
                mainGain.gain.setValueAtTime(0, audioCtx.currentTime);
                mainGain.gain.linearRampToValueAtTime(0.5, audioCtx.currentTime + 0.1);
                mainGain.gain.setValueAtTime(0.5, audioCtx.currentTime + 0.8);
                mainGain.gain.exponentialRampToValueAtTime(0.01, audioCtx.currentTime + 1.2);

                lfo.connect(tremolo.gain);
                osc.connect(tremolo);
                tremolo.connect(mainGain);
                
                // Route to both direct output and reverb
                mainGain.connect(audioCtx.destination);
                mainGain.connect(convolver);
                
                lfo.start(); osc.start();
                lfo.stop(audioCtx.currentTime + 1.3); osc.stop(audioCtx.currentTime + 1.3);
            }

            function playMoveSound() {
                const osc = audioCtx.createOscillator();
                const gainNode = audioCtx.createGain();
                
                osc.type = 'sine';
                osc.frequency.setValueAtTime(300, audioCtx.currentTime);
                osc.frequency.exponentialRampToValueAtTime(500, audioCtx.currentTime + 0.05);
                
                gainNode.gain.setValueAtTime(0.4, audioCtx.currentTime);
                gainNode.gain.exponentialRampToValueAtTime(0.001, audioCtx.currentTime + 0.05);
                
                osc.connect(gainNode);
                
                // Route to direct output AND reverb to create the echo
                gainNode.connect(audioCtx.destination);
                gainNode.connect(convolver);
                
                osc.start();
                osc.stop(audioCtx.currentTime + 0.05);
            }

            function playVictorySound() {
                const notes = [261.63, 329.63, 392.00, 523.25]; // C4, E4, G4, C5 Fanfare
                const startTime = audioCtx.currentTime;
                
                notes.forEach((freq, index) => {
                    const osc = audioCtx.createOscillator();
                    const gain = audioCtx.createGain();
                    osc.type = 'triangle';
                    osc.frequency.value = freq;
                    
                    const time = startTime + index * 0.15;
                    const duration = index === notes.length - 1 ? 2.0 : 0.15;
                    
                    gain.gain.setValueAtTime(0, time);
                    gain.gain.linearRampToValueAtTime(0.3, time + 0.05);
                    gain.gain.exponentialRampToValueAtTime(0.01, time + duration);
                    
                    osc.connect(gain);
                    gain.connect(audioCtx.destination);
                    gain.connect(convolver);
                    
                    osc.start(time);
                    osc.stop(time + duration);
                });
            }

            // --- 3. ANIMATION SEQUENCE ---
            // Reset visuals
            for (let i = 0; i < maxSteps; i++) {
                const line = document.getElementById('line-__UID__-' + i);
                const circle = document.getElementById('circle-__UID__-' + i);
                if (line) line.setAttribute('opacity', '0');
                if (circle) circle.setAttribute('opacity', '0');
            }

            // Play the opening neigh, then wait 1.5 seconds before starting the tour
            playNeigh();
            
            setTimeout(() => {
                let currentStep = 0;
                const intervalId = setInterval(() => {
                    if (currentStep >= maxSteps) {
                        clearInterval(intervalId);
                        playVictorySound();
                        btn.style.background = '#4CAF50';
                        btn.innerText = '▶ Replay Tour';
                        isPlaying___UID__ = false;
                        return;
                    }
                    
                    // Reveal SVG paths
                    const line = document.getElementById('line-__UID__-' + currentStep);
                    const circle = document.getElementById('circle-__UID__-' + currentStep);
                    if (line) line.setAttribute('opacity', '1');
                    if (circle) circle.setAttribute('opacity', '1');
                    
                    playMoveSound();
                    currentStep++;
                    
                }, delay);
            }, 1500); // 1.5 second pause for the horse neigh
        }
    </script>
    </div>|}

let show_solution ?(width = "50%") (board : int array array) : unit =
  let n = Array.length board in
  let cell_size  = 50 in
  let total_size = n * cell_size in
  let path = Array.make (n * n) (0, 0) in
  Array.iteri (fun r row -> Array.iteri (fun c k -> path.(k) <- (r, c)) row) board;
  let uid  = Printf.sprintf "%07x" (Random.bits ()) in
  let html = Buffer.create 16384 in
  Buffer.add_string html (Printf.sprintf "<div id=\"tour-container-%s\">" uid);
  Buffer.add_string html (replace_all button "__UID__" uid);
  Buffer.add_string html
    (Printf.sprintf "<svg width=\"%s\" viewBox=\"0 0 %d %d\" xmlns=\"http://www.w3.org/2000/svg\" style=\"font-family: sans-serif;\">"
       width total_size total_size);
  (* Draw board *)
  for r = 0 to n - 1 do
    for c = 0 to n - 1 do
      let is_black = (r + c) mod 2 <> 0 in
      let color = if is_black then "#b58863" else "#f0d9b5" in
      let x = c * cell_size and y = r * cell_size in
      Buffer.add_string html
        (Printf.sprintf "<rect x=\"%d\" y=\"%d\" width=\"%d\" height=\"%d\" fill=\"%s\" />" x y cell_size cell_size color);
      Buffer.add_string html
        (Printf.sprintf "<text x=\"%d\" y=\"%d\" text-anchor=\"middle\" font-size=\"14\" fill=\"%s\">%d</text>"
           (x + cell_size / 2) (y + cell_size / 2 + 5) (if is_black then "#f0d9b5" else "#b58863") board.(r).(c))
    done
  done;
  (* Start position *)
  let (start_r, start_c) = path.(0) in
  Buffer.add_string html
    (Printf.sprintf "<circle cx=\"%d\" cy=\"%d\" r=\"8\" fill=\"darkblue\" />"
       (start_c * cell_size + cell_size / 2) (start_r * cell_size + cell_size / 2));
  (* Path lines and dots *)
  for k = 0 to n * n - 2 do
    let (r1, c1) = path.(k) and (r2, c2) = path.(k + 1) in
    let x1 = c1 * cell_size + cell_size / 2 and y1 = r1 * cell_size + cell_size / 2 in
    let x2 = c2 * cell_size + cell_size / 2 and y2 = r2 * cell_size + cell_size / 2 in
    Buffer.add_string html
      (Printf.sprintf "<line id=\"line-%s-%d\" x1=\"%d\" y1=\"%d\" x2=\"%d\" y2=\"%d\" stroke=\"darkblue\" stroke-width=\"3\" stroke-opacity=\"0.6\" opacity=\"0\"></line>"
         uid k x1 y1 x2 y2);
    Buffer.add_string html
      (Printf.sprintf "<circle id=\"circle-%s-%d\" cx=\"%d\" cy=\"%d\" r=\"3\" fill=\"darkblue\" opacity=\"0\"></circle>" uid k x2 y2)
  done;
  Buffer.add_string html "</svg>";
  (* Animation and Audio Logic *)
  Buffer.add_string html
    (replace_all (replace_all script "__UID__" uid) "__MAXSTEPS__" (string_of_int (n * n - 1)));
  ignore (Jupyter_notebook.display "text/html" (Buffer.contents html))

In [ ]:
show_solution board